# DICOM

In [ ]:
import os
import numpy as np
import pydicom
import plotly.express as px

path_ = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"
read_dcm = lambda i : pydicom.dcmread(os.path.join(path_, i)).pixel_array
ct = list(map(read_dcm, sorted(os.listdir(path_))))
ct = np.stack(ct, axis=0)
print(ct.shape)

fig = px.imshow(
    ct, 
    animation_frame=0,
    binary_string=True, 
    labels=dict(animation_frame="slice"))
fig.show()

In [ ]:
import glob
import pydicom

path_ = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"
for i in glob.glob(os.path.join(path_, "*")):
    dcm = pydicom.dcmread(i)
    # x, y, z
    print(dcm.get((0x0020, 0x0032)).value)

In [ ]:
# (0008, 0070) Manufacturer
import os, glob

global Manufacturer
Manufacturer = []

def print_recursiverly(path):
    try:
        is_dcm = all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))])
    except PermissionError:
        is_dcm = False
    
    if is_dcm:
        if len(glob.glob(os.path.join(path, "*"))) > 0:
            dcm = pydicom.dcmread(glob.glob(os.path.join(path, "*"))[0])
            if dcm.get((0x0008, 0x0060)).value == "CT":
                value = dcm.get((0x0008, 0x0070)).value
                print(value)
                type = "CBCT" if value == "ELEKTA" else "CT"
                Manufacturer.append({"type": type, "value": value, "path": path})
    else:
        for j in glob.glob(os.path.join(path, "*")):
            print_recursiverly(j)

for patient in glob.glob(os.path.join(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data", "*")):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i)

In [ ]:
for i in df["value"].unique():
    print(i)
    dff = df[df["value"] == i]
    for i, row in dff.iterrows():
        print(row["path"])

In [ ]:
import pandas
import plotly.express as px

df = pandas.DataFrame(Manufacturer)
for type in df["type"].unique():
    dff = df[df["type"] == type]
    dff = [{"type": k, "value": len(dff[dff["value"] == k])} for k in dff["value"].unique()]
    dff = pandas.DataFrame(dff)
    print(dff)

    fig = px.pie(dff, values='value', names='type')
    fig.show()

In [ ]:
from datetime import datetime

datetime_object = datetime.strptime(f[0x0008, 0x0012].value, '%Y%m%d')
print(datetime_object.day)

In [ ]:
import glob, os

for i in glob.glob(os.path.join(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\004", "CT*")):
    print(i)
    for j in glob.glob(os.path.join(i, "*")):
        print(j)
        for k in glob.glob(os.path.join(j, "*"))[:1]:
            if pydicom.misc.is_dicom(k):
                dcm = pydicom.dcmread(k)
                print(datetime.strptime(dcm[0x0008, 0x0012].value, '%Y%m%d'))

In [ ]:
global XCURRENT
XCURRENT = []

def print_recursiverly(path):
    try:
        is_dcm = all([pydicom.misc.is_dicom(j) for j in glob.glob(os.path.join(path, "*"))])
    except PermissionError:
        is_dcm = False
    
    if is_dcm:
        if len(glob.glob(os.path.join(path, "*"))) > 0:
            dcm = pydicom.dcmread(glob.glob(os.path.join(path, "*"))[0])
            if dcm.get((0x0008, 0x0060)).value == "CT":
                type = "CBCT" if "CBCT" in os.path.split(path)[1] else "CT"
                value = dcm.get((0x0018, 0x1151)).value
                print(value)
                XCURRENT.append({"type": type, "value": value})
    else:
        for j in glob.glob(os.path.join(path, "*")):
            print_recursiverly(j)

for patient in glob.glob(os.path.join(r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data", "*")):
    print(patient)
    for i in glob.glob(os.path.join(patient, "*")):
        print_recursiverly(i)

In [ ]:
import pandas
import plotly.express as px
import plotly.figure_factory as ff

df = pandas.DataFrame(XCURRENT)

group_labels = ['CT', 'CBCT']
hist_data = [df[df["type"] == i]["value"] for i in group_labels]
fig = ff.create_distplot(hist_data, group_labels)
fig.show()

The tag (0020,0052) Frame of Reference UID is available in three different ways:
    (0020,0052) Frame of Reference UID
    (3006,0010) -> item -> (0020,0052) Frame of Reference UID
    (3006,0020) -> item -> (3006,0024) Referenced Frame of Reference UID

# clinical

In [ ]:
import pandas

path_ = r"c:\Users\bguet\Desktop\data\ARTIX\toxicity_data\20241021_EFFICACY_LTSI.csv"

df = pandas.read_csv(path_, delimiter=";")
df

In [ ]:
for i in df["XERO12_IMPUT1"].unique():
    if pandas.isna(i):
        n = len(df[pandas.isna(df["XERO12_IMPUT1"])])
    else:
        n = len(df[df["XERO12_IMPUT1"] == i])
    print(f"{i}: {n} ({100*n/len(df):.0f}%)")

In [ ]:
import pandas

df = pandas.read_csv(r"c:\Users\bguet\Desktop\data\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv", sep=";")
df = df[df["USUBJID"] == df["USUBJID"].unique()[2]]
df = df[df["MEASTYP"] == "Stimulated salivation flow"]
df

In [ ]:
df.columns

# create ARTIX

In [1]:
import glob, os, tqdm
import artix

path = r"E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data"
patients = []
for p in tqdm.tqdm(glob.glob(os.path.join(path, "*"))[:5], ncols=50):
    patients.append(artix.load_patient(
        path=p,
        id_map=r"E:\bilel\data\ARTIX\ARTIX\ARTIX_ID_CORRELATION.xlsx",
        clinical_csv=[
            r"E:\bilel\data\ARTIX\ARTIX\toxicity_data\20241021_PATIENT_DESCRIPTION_LTSI.csv",
            r"E:\bilel\data\ARTIX\ARTIX\toxicity_data\20241021_EFFICACY_LTSI.csv",
            r"E:\bilel\data\ARTIX\ARTIX\toxicity_data\20241021_SALIVATION_DATA_LTSI.csv",
            r"E:\bilel\data\ARTIX\ARTIX\toxicity_data\20241021_TREATMENT_LTSI.csv",
        ],
        log=r"C:\Users\bilel.guetarni\Desktop\workspace\RT-tox\ARTIX\log",
        ))

print(len(patients))

  0%|                       | 0/5 [00:00<?, ?it/s]

total data loaded from E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\001
<class 'dicom_class.CBCT'>:	 4
<class 'dicom_class.RTDOSE'>:	 11
<class 'dicom_class.CT'>:	 6
<class 'dicom_class.RTSTRUCT'>:	 6


 20%|███            | 1/5 [00:52<03:28, 52.14s/it]

total data loaded from E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\002
<class 'dicom_class.CBCT'>:	 3
<class 'dicom_class.RTDOSE'>:	 9
<class 'dicom_class.CT'>:	 6
<class 'dicom_class.RTSTRUCT'>:	 6


 40%|██████         | 2/5 [01:43<02:34, 51.55s/it]

...

total data loaded from E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\004
<class 'dicom_class.CBCT'>:	 7
<class 'dicom_class.RTDOSE'>:	 11
<class 'dicom_class.CT'>:	 6
<class 'dicom_class.RTSTRUCT'>:	 6


 80%|████████████   | 4/5 [03:15<00:49, 49.03s/it]

total data loaded from E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\005
<class 'dicom_class.CBCT'>:	 7
<class 'dicom_class.RTDOSE'>:	 11
<class 'dicom_class.CT'>:	 6
<class 'dicom_class.RTSTRUCT'>:	 7


100%|███████████████| 5/5 [04:35<00:00, 55.10s/it]

5


In [2]:
len(patients)

5

In [6]:
p = patients[0]
print(p.id)
print(len(p.ct))
print(len(p.cbct))

print(p.clinical)

98
6
4
{'age': 49, 'sexe': 'Male', 'hpv': 'Negative', 'arm': 'Replanification Arm', 'inclusion_dt': '12/06/2017', 'randomization_dt': '20/06/2017', 'is_prog_recc': 'No', 'progression_dt': nan, 'Stimulated salivation flow (Inclusion)': '2820', 'Stimulated salivation flow (M6)': '1580', 'Stimulated salivation flow (M12)': '1440', 'Stimulated salivation flow (M18)': '1280', 'Stimulated salivation flow (M24)': '980', 'start_treatment_dt': '04/07/2017'}


In [ ]:
import os
subpath = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data"
path = r"c:\Users\bguet\Desktop\data\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000"

os.path.relpath(path, start=subpath)

## check segmentation

In [ ]:
import dicom_utils
import numpy as np

OAR_to_seg = {7: "parotid_gland_left", 8: "parotid_gland_right", 9: "submandibular_gland_right", 10: "submandibular_gland_left"}
threshold = 0.1

for p in patients:
    print(p.id)
    
    for ct in p.ct:
        print(ct.path)
        if ct.rtstruct is None:
            continue

        # TotalSegmentator must be applied before reloading nii
        out = ct.apply_totalsegmentator()

        ct.load_nii()

        for structure_ID, structure_name in OAR_to_seg.items():
            print(structure_name)
            struct_mask = (out == structure_ID).astype("uint8")

            d = 0
            best_oar = None
            for oar in ct.rtstruct.get_all_OARs():
                voxel_coords = ct.rtstruct.get_contours(oar)
                original_mask = dicom_utils.fill_vol_ctrs(struct_mask.shape, voxel_coords)

                dice = 2 * (np.logical_and(struct_mask, original_mask).sum()) / (struct_mask.sum() + original_mask.sum())
                if dice > d:
                    d = dice
                    best_oar = oar
            print(f"{d:.2f} {best_oar}")

98
E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\001\CT0\DOE^JOHN_ANON61841_CT_2017-06-21_100509_ORL.(sauf.sinus)_ORL.2MM_n198__00000
No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.


c:\Users\bilel.guetarni\anaconda3\envs\radiomics\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


parotid_gland_left
0.33 Parotide_homolaterale
parotid_gland_right
0.32 Parotide_controlaterale
submandibular_gland_right
0.56 Sous_max_controlateral
submandibular_gland_left
0.64 Sous_max_homolateral
E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\001\CT1\DOE^JOHN_ANON61841_CT_2017-07-10_114626_ORL.(sauf.sinus)_ORL.2MM_n219__00000
No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.
parotid_gland_left
0.32 Parotide_homolaterale
parotid_gland_right
0.33 Parotide_controlaterale
submandibular_gland_right
0.50 Sous_max_controlateral
submandibular_gland_left
0.54 Sous_max_homolateral
E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\001\CT2\DOE^JOHN_ANON61841_CT_2017-07-17_114513_ORL.(sauf.sinus)_ORL.2MM_n201__00000
No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.
parotid_gland_left
0.34 Parotide_homolaterale
parotid_gland_right
0.32 Parotide_controlatera

AttributeError: 'Dataset' object has no attribute 'ContourSequence'

In [7]:
import pydicom

for p in patients:
    for ct in p.ct:
        if ct.rtstruct is None:
            continue

        dcm = pydicom.dcmread(ct.rtstruct.get_dcm_path())
        for roi_contour in dcm.ROIContourSequence:
            if not hasattr(roi_contour, "ContourSequence"):
                print(f"[NO]\t {ct.rtstruct.path}")

[NO]	 E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\004\CT4\DOE^JOHN_ANON17151_RTst_2017-05-22_123032_ORL.(sauf.sinus)_CT.4.DOSI.0_n1__00000
[NO]	 E:\bilel\data\ARTIX\ARTIX\DICOM_ARTIX_data\004\CT5\DOE^JOHN_ANON17151_RTst_2017-05-29_120721_ORL.(sauf.sinus)_CT.5.DOSI.0_n1__00000
